This notebook contains data analysis, and all relevant preprocessing for the deployed database, as well as some preprocessing for the model training.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import krippendorff
from scipy.stats import norm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from ppi_py import ppi_mean_pointestimate, ppi_mean_ci, ppi_logistic_pointestimate, ppi_logistic_ci

import json, codecs, os, ast, pickle

from utils import translate_labels

### Merging scraped articles

Merging all the articles from each sector/query, and dropping duplicates (duplicate title and urls).

There is a small overlap of same articles in different sectors, so these articles get several sectors, instead of being dropped as duplicates.

In [5]:
def merge_scraped_articles(directory):
    dfs = []
    for filename in os.listdir(directory):  
        sector = filename.split("_")[0]
        df = pd.read_json(os.path.join(directory, filename), lines=True)
        df[f'sector_{sector}'] = True
        dfs.append(df)

    final_df = pd.concat(dfs, ignore_index=True)
    sector_cols = [c for c in final_df.columns if c.startswith('sector_')]
    final_df[sector_cols] = final_df[sector_cols].fillna(False)

    agg_funcs = {col: 'max' if col.startswith('sector_') else 'first' for col in final_df.columns}
    
    final_df = final_df.groupby('title', as_index=False).agg(agg_funcs)
    final_df = final_df.groupby('url', as_index=False).agg(agg_funcs)

    return final_df.reset_index(drop=True)

In [7]:
english_dataset = merge_scraped_articles("datasets/raw/english")
english_dataset["an"]=english_dataset["document_id"]
english_dataset.to_csv("datasets/raw/english.csv") #used as input by labeller.py

In [8]:
dutch_dataset = merge_scraped_articles("datasets/raw/dutch")
dutch_dataset["an"] = dutch_dataset["document_id"]
dutch_dataset.to_csv("datasets/raw/dutch.csv") #used as input by labeller.py

### Merging QuotaClimat data + scraped data

Loading QuotaClimat data and human+LLM annotations

keyword filtering following QuotaClimat's information on what is most likely to be an environmental article

In [9]:
qc_dataset_full = pd.read_csv("datasets/raw/french/french_full.csv")
qc_dataset = qc_dataset_full[(qc_dataset_full["number_of_ressources_no_hrfp"]>=7) | (qc_dataset_full["number_of_biodiversite_no_hrfp"]>=7) | (qc_dataset_full["number_of_climat_no_hrfp"]>=7)]
del qc_dataset_full

remove duplicates, NaNs in title/body, unicode ids of whitespaces, filters by word count

In [10]:
qc_dataset = qc_dataset[qc_dataset["duplicate_status"] != "DUP"]
qc_dataset = qc_dataset.dropna(subset=["title", "body"])
qc_dataset['body'] = qc_dataset['body'].str.replace('\xa0', ' ')
qc_dataset['word_count'] = qc_dataset['body'].str.split().str.len()
qc_dataset = qc_dataset[qc_dataset["word_count"] > 100]

In [11]:
qc_frame_labels = pd.read_csv("datasets/llm_labelled/french_frames.csv")
qc_angle_labels = pd.read_csv("datasets/llm_labelled/french_angles.csv")

annotator_1 = pd.read_csv("datasets/annotated/human_labelled1.csv")
annotator_2 = pd.read_csv("datasets/annotated/human_labelled2.csv")
annotated = pd.read_csv("datasets/annotated/human_agreement.csv", header=1)[["an","frame_french"]]

english_frame_labelled = pd.read_csv("datasets/llm_labelled/english_frames.csv")
english_angle_labelled = pd.read_csv("datasets/llm_labelled/english_angles.csv")

dutch_frame_labelled = pd.read_csv("datasets/llm_labelled/dutch_frames.csv")
dutch_angle_labelled = pd.read_csv("datasets/llm_labelled/dutch_angles.csv")

Probabilities and attention from RoBERTa model:

In [12]:
with open("datasets/attention/attention_frames.pkl","rb") as f:
    attention_frames = pickle.load(f)
with open("datasets/attention/attention_angles.pkl","rb") as f:
    attention_angles = pickle.load(f)

##### Quota Climat preprocessing

Merging original quota climat dataset with the LLM-labelled frames and angles

In [13]:
qc_df = qc_dataset.merge(qc_frame_labels[["an","frames"]],on="an").merge(qc_angle_labels[["an","angles"]],on="an")

Stages in QC dataset are only available per domain, so merging across domains

In [14]:
qc_df['stage_concept'] = qc_df['predict_climat_constat'] | qc_df['predict_biodiversite_constat'] | qc_df['predict_ressources_constat']
qc_df['stage_cause'] = qc_df['predict_climat_cause'] | qc_df['predict_biodiversite_cause']
qc_df['stage_consequence'] = qc_df['predict_climat_consequence'] | qc_df['predict_biodiversite_consequence']
qc_df['stage_solution'] = qc_df['predict_climat_solution'] | qc_df['predict_biodiversite_solution'] | qc_df['predict_ressources_solution']

Standardising outlet names:

In [15]:
qc_df = qc_df.rename(columns={'predict_agriculture_alimentation':'sector_agriculture','predict_mobilite':'sector_transport','predict_batiments_amenagement':'sector_buildings',
                    'predict_economie_circulaire':'sector_resource','predict_energie':'sector_energy','predict_industrie':'sector_industry','predict_eau':'sector_water',
                    'predict_ecosysteme':'sector_ecosystem','predict_climat':'domain_climate','predict_biodiversite':'domain_biodiversity','predict_ressources':'domain_ressources'})

qc_df["outlet"] = qc_df["source_name"]
qc_df["published_at"] = qc_df["publication_date"]
qc_df["language"] = "french"

outlet_rename = {
    # Le Monde
    'Le Monde.fr': 'Le Monde',
    'Le Monde': 'Le Monde',
    'M, le magazine du Monde': 'Le Monde',
    
    # Le Figaro
    'Le Figaro': 'Le Figaro',
    'lefigaro.fr': 'Le Figaro',
    'Le Figaro Magazine': 'Le Figaro',
    'Le Figaro Premium': 'Le Figaro',
    
    # Libération
    'Libération': 'Libération',
    'Libération.fr': 'Libération',
    
    # Le Parisien / Aujourd'hui en France
    'Le Parisien': 'Le Parisien',
    'leParisien.fr': 'Le Parisien',
    "Aujourd'hui en France": 'Le Parisien',
    "Le Parisien Week-End - Aujourd'hui en France Week-End": 'Le Parisien',
    
    # La Croix
    'La Croix': 'La Croix',
    'La-Croix.com': 'La Croix',
    
    # Les Echos
    'Les Echos': 'Les Echos',
    'lesechos.fr': 'Les Echos',
    'Les Echos.fr': 'Les Echos',
    
    # La Tribune
    'La Tribune': 'La Tribune',
    'latribune.fr': 'La Tribune',
    'La Tribune.fr': 'La Tribune',
    
    # Le Point
    'Le Point': 'Le Point',
    'lepoint.fr': 'Le Point',
    'LePoint.fr': 'Le Point',
    
    # L'Obs / Nouvel Obs
    "L'Obs": "L'Obs",
    'Nouvelobs.com': "L'Obs",
    
    # Le HuffPost
    'Le HuffPost': 'Le HuffPost',
    
    # L'Humanité
    "L'Humanité": "L'Humanité",
    "L'Humanité Dimanche": "L'Humanité",
    
    # Mediapart
    'mediapart.fr': 'Mediapart',
    'Mediapart': 'Mediapart',
    
    # Marianne
    'Marianne': 'Marianne',
    'Marianne.net': 'Marianne',
    
    # JDD
    'Lejdd.fr': 'Le Journal du Dimanche',
    
    # 20 Minutes
    '20Minutes.fr': '20 Minutes',
    
    # France 24
    'France24.com': 'France 24',
    
    # AFP
    'Agence France Presse': 'Agence France Presse',
    
    # Ouest-France
    'Ouest-France': 'Ouest-France',
    'Ouest-France.fr': 'Ouest-France',
    
    # Sud Ouest
    'Sud Ouest': 'Sud Ouest',
    'Sudouest.fr': 'Sud Ouest',
    
    # La Montagne
    'La Montagne': 'La Montagne',
    'Lamontagne.fr': 'La Montagne',
    
    # La Dépêche du Midi
    'La Dépêche du Midi': 'La Dépêche du Midi',
    
    # L'Indépendant
    "L'Indépendant": "L'Indépendant",
    
    # Midi Libre
    'Midi Libre': 'Midi Libre',
    
    # Centre Presse
    'Centre Presse': 'Centre Presse',
    
    # La Provence
    'La Provence': 'La Provence',
    
    # Nice-Matin
    'Nice-Matin': 'Nice-Matin',
    
    # Var-Matin
    'Var-Matin': 'Var-Matin',
    
    # Corse-Matin
    'Corse-Matin': 'Corse-Matin',
    
    # L'Est Républicain
    "L'Est Républicain": "L'Est Républicain",
    
    # Le Républicain Lorrain
    'Le Républicain Lorrain': 'Le Républicain Lorrain',
    
    # Vosges Matin
    'Vosges Matin': 'Vosges Matin',
    
    # L'Yonne Républicaine
    'L´Yonne Républicaine': "L'Yonne Républicaine",
    'Lyonne.fr': "L'Yonne Républicaine",
    
    # L'Echo Républicain
    'L´Echo Républicain': "L'Echo Républicain",
    'Lechorepublicain.fr': "L'Echo Républicain",
    
    # L'Union / L'Ardennais
    "L'Union / L'Ardennais": "L'Union / L'Ardennais",
    
    # La Charente Libre
    'La Charente Libre': 'La Charente Libre',
    
    # Le Journal du Centre
    'Le Journal du Centre': 'Le Journal du Centre',
    'Lejdc.fr': 'Le Journal du Centre',
    
    # La Voix du Nord
    'La Voix du Nord': 'La Voix du Nord',
    
    # Nord Eclair
    'Nord Eclair': 'Nord Eclair',
    
    # Nord Littoral
    'Nord Littoral': 'Nord Littoral',
    
    # Courrier Picard
    'Courrier Picard': 'Courrier Picard',
    
    # L'Aisne Nouvelle
    "L'Aisne Nouvelle": "L'Aisne Nouvelle",
    
    # Le Progrès
    'Le Progrès': 'Le Progrès',
    
    # Le Berry Républicain
    'Le Berry Républicain': 'Le Berry Républicain',
    'Leberry.fr': 'Le Berry Républicain',
    
    # Le Populaire du Centre
    'Le Populaire du Centre': 'Le Populaire du Centre',
    'Lepopulaire.fr': 'Le Populaire du Centre',
    
    # La République du Centre
    'La République du Centre': 'La République du Centre',
    'Larep.fr': 'La République du Centre',
    
    # La Nouvelle République du Centre Ouest
    'La Nouvelle République du Centre Ouest': 'La Nouvelle République du Centre Ouest',
    
    # La République des Pyrénées
    'La République des Pyrénées': 'La République des Pyrénées',
    
    # La Nouvelle République des Pyrénées
    'La Nouvelle République des Pyrénées': 'La Nouvelle République des Pyrénées',
    
    # L'Eclair des Pyrénées
    "L'Eclair des Pyrénées": "L'Eclair des Pyrénées",
    
    # Le Petit Bleu de Lot-et-Garonne
    'Le Petit Bleu de Lot-et-Garonne': 'Le Petit Bleu de Lot-et-Garonne',
    
    # Dordogne Libre
    'Dordogne Libre': 'Dordogne Libre',
    
    # Le Télégramme
    'Le Télégramme': 'Le Télégramme',
}

qc_df['outlet'] = qc_df['outlet'].replace(outlet_rename)
qc_df[["subject","industry","geographic"]] = None

Translating frame/angle names into English, so they are consistent with the scraped data:

In [16]:
qc_df["frames"] = qc_df["frames"].apply(lambda x: translate_labels(x,"frames","french"))
qc_df["angles"] = qc_df["angles"].apply(lambda x: translate_labels(x,"angles","french"))

##### Scraped data preprocessing

In [17]:
english_dataset["an"] = english_dataset["document_id"].copy()
english_df = english_dataset.merge(english_frame_labelled[["an","frames"]],on="an")
english_df = english_df.merge(english_angle_labelled[["an","angles"]],on="an")
english_df["language"] = "english"
english_df["frames"] = english_df["frames"].apply(ast.literal_eval)
english_df["angles"] = english_df["angles"].apply(ast.literal_eval)

english_df["subject"] = english_df["classification"].apply(lambda x: x.get("Subject",None))
english_df["industry"] = english_df["classification"].apply(lambda x: x.get("Industry",None))
english_df["geographic"] = english_df["classification"].apply(lambda x: x.get("Geographic",None))

In [29]:
#standardising outlet names
english_outlet_rename = {
    'The Guardian - Final Edition': 'The Guardian',
    'Guardian.com': 'The Guardian',
    'The Guardian (London)': 'The Guardian',
 
    'The Observer (archive from the Guardian)': 'The Observer',
    'The Observer - London (Print)': 'The Observer',
 
    'thetimes.co.uk': 'The Times (London)',
    'Times Online': 'The Times (London)',
 
    'telegraph.co.uk': 'The Daily Telegraph (London)',
 
    'The Independent - Daily Edition': 'The Independent (United Kingdom)',
 
    'MailOnline': 'The Daily Mail (London)',
    'DAILY MAIL (London)': 'The Daily Mail (London)',
 
    'mirror.co.uk': 'Daily Mirror',
 
    'FT.com': 'Financial Times (London, England)',
 
    'The Times of India (TOI)': 'The Times of India',
    'Times of India (Electronic Edition)': 'The Times of India',
 
    'USA Today Online': 'USA Today',
 
    'The Herald (Scotland) Online': 'The Herald (Glasgow)',
    'The Herald (Scotland) Print': 'The Herald (Glasgow)',

    'liverpoolecho.co.uk': 'Liverpool Echo',
    'manchestereveningnews.co.uk': 'Manchester Evening News',
    'chesterchronicle.co.uk': 'Chester Chronicle',
    'birminghammail.co.uk': 'Birmingham Evening Mail',
 
    'South China Morning Post.com': 'South China Morning Post',
 
    'thestar.com': 'The Toronto Star',
 
    'Jpost.com (The Jerusalem Post online edition)': 'Jerusalem Post',
 
    'The Peterborough Examiner (Ontario) Online': 'The Peterborough Examiner (Ontario)',
 
    'The Age (Melbourne, Australia) - Online': 'The Age (Melbourne, Australia)',
 
    'The Sydney Morning Herald (Australia) - Online': 'The Sydney Morning Herald',
 
    'Canberra Times (Australia)': 'The Canberra Times',
 
    'Belfast Telegraph Online': 'Belfast Telegraph',
 
    'Business Standard (Online)': 'Business Standard',
 
    'The Daily Monitor': 'Daily Monitor',

    'This Day': 'This Day (Lagos)',
    'CXOtoday.com': 'CXOToday.com',
    'Business World': 'BusinessWorld',                             
    'Azernews - News from Azerbaijan, Business, Energy, Analysis': 'Azer News',
    'Express Online': 'The Express',                             
    'adelaidenow Online': 'The Advertiser (Australia)',                
    'Tanzania Daily News': 'Daily News (Tanzania)',                   
    'TendersInfo - Contract Awards': 'TendersInfo',                 
    'Time Online': 'The Times (London)'                      
}

english_df["outlet"] = english_df["outlet"].replace(english_outlet_rename)

In [24]:
dutch_dataset["an"] = dutch_dataset["document_id"].copy()
dutch_df = dutch_dataset.merge(dutch_frame_labelled[["an","frames"]],on="an")
dutch_df = dutch_df.merge(dutch_angle_labelled[["an","angles"]],on="an")
dutch_df["language"] = "dutch"
dutch_df["frames"] = dutch_df["frames"].apply(ast.literal_eval)
dutch_df["angles"] = dutch_df["angles"].apply(ast.literal_eval)

dutch_df["subject"] = dutch_df["classification"].apply(lambda x: x.get("Subject",None))
dutch_df["industry"] = dutch_df["classification"].apply(lambda x: x.get("Industry",None))
dutch_df["geographic"] = dutch_df["classification"].apply(lambda x: x.get("Geographic",None))

dutch_df["frames"]=dutch_df["frames"].apply(lambda x: translate_labels(x,"frames","dutch"))
dutch_df["angles"]=dutch_df["angles"].apply(lambda x: translate_labels(x,"angles","dutch"))

In [33]:
#standardising outlet names
dutch_outlet_rename = {
    'AD/Algemeen Dagblad.nl': 'AD/Algemeen Dagblad',
    'Trouw.nl': 'Trouw',
    'De Volkskrant.nl': 'de Volkskrant',        
    'De Telegraaf.nl': 'De Telegraaf',
    'NRC': 'NRC Handelsblad', 
    'NRC.nl': 'NRC Handelsblad',

    'PZC.nl': 'Provinciale Zeeuwse Courant', 
    'Tubantia.nl': 'De Twentsche Courant Tubantia',
    'Dagblad van het Noorden.nl': 'Dagblad van het Noorden',
    'Leeuwarder Courant.nl': 'Leeuwarder Courant',
    'Noordhollands Dagblad.nl': 'Noordhollands Dagblad',
    'Haarlems Dagblad.nl': 'Haarlems Dagblad',
    'de limburger.nl': 'De Limburger',
    'Dagblad De Limburger.nl': 'De Limburger',
    'Eindhovens Dagblad.nl': 'Eindhovens Dagblad',
    'Leidsch Dagblad.nl': 'Leidsch Dagblad',
    'De Gooi- en Eemlander.nl': 'De Gooi- en Eemlander',
    'Friesch Dagblad.nl': 'Friesch Dagblad',
    'IJmuider Courant.nl': 'IJmuider Courant',
    'De Stentor.nl': 'De Stentor',
    'De Gelderlander.nl': 'De Gelderlander',
    'Brabants Dagblad.nl': 'Brabants Dagblad',
    'Het Parool.nl': 'Het Parool',
    'BN De Stem.nl': 'BN/DeStem',

    'libelle.nl': 'Libelle.nl',   
    'flair.nl': 'Flair',
    'Melkvee.nl': 'Melkvee',
    'Vee-en-Gewas.nl': 'Vee & Gewas',
    'Agraaf.nl': 'Agraaf',
    'Investing.com (Dutch) Nederlands Edition': 'Investing.com (Dutch)',

    'AD/Rotterdams Dagblad': 'AD/Algemeen Dagblad',
    'AD/Haagsche Courant': 'AD/Algemeen Dagblad',
    'AD/Groene Hart': 'AD/Algemeen Dagblad',
    'AD/Utrechts Nieuwsblad': 'AD/Algemeen Dagblad',
    'AD/Rivierenland': 'AD/Algemeen Dagblad',
    'AD/Amersfoortse Courant': 'AD/Algemeen Dagblad',
    'AD/De Dordtenaar': 'AD/Algemeen Dagblad',
    'Flevopost Lelystad': 'Flevopost',
    'Flevopost Dronten': 'Flevopost',

     'WebNews - Dutch\nBinnenlands Bestuur\nhttps://www.binnenlandsbestuur.nl/ruimte-en-milieu/tata-steel-nederland/het-groen-staal-project-wat-is-dat-eigenlijk\n22 April 2026 Woensdag':'Binnenlands Bestuur'
}
dutch_df["outlet"] = dutch_df["outlet"].replace(dutch_outlet_rename)

Storing these for model training:

In [37]:
qc_df.to_csv("datasets/training/french_labelled.csv")
english_df.to_csv("datasets/training/english_labelled.csv")
dutch_df.to_csv("datasets/training/dutch_labelled.csv")

In [38]:
all_data = pd.concat([qc_df,english_df,dutch_df])

In [39]:
mlb = MultiLabelBinarizer()

encoded_df_frames = pd.DataFrame(mlb.fit_transform(all_data['frames']), 
    columns=mlb.classes_, 
    index=all_data.index).astype(bool)

encoded_df_angles = pd.DataFrame(mlb.fit_transform(all_data['angles']), 
    columns=mlb.classes_, 
    index=all_data.index).astype(bool)

all_data = pd.concat([all_data, encoded_df_frames, encoded_df_angles], axis=1)

These are for matching the database schema:

In [40]:
all_data["id"]=None
all_data["source_id"] = all_data["an"]
all_data["query_source"] = None
all_data['classified_at'] = None
all_data['ingested_at'] = None
all_data["published_at"] = all_data["published_at"].apply(lambda x: str(x)[:10])
frame_rename = {"Conflict":"frame_conflict","Economic Consequences":"frame_economic",
                                "Human Interest":"frame_human_interest","Morality":"frame_morality","Responsibility":"frame_attribution"}
all_data = all_data.rename(columns=frame_rename)
angle_rename = {"Economics":"angle_economics","Geopolitics":"angle_geopolitics","Health":"angle_health",
                               "Jobs":"angle_jobs","Politics":"angle_politics","Purchasing Power":"angle_purchasing_power"}
all_data = all_data.rename(columns=angle_rename)

In [41]:
all_data["word_count"] = (all_data["title"]+all_data["body"]).apply(lambda x: len(x.split(" ")))

In [42]:
all_data["attention_frames"] = attention_frames
all_data["attention_angles"] = attention_angles

Getting the RoBERTa model probabilities, as well as the indexes with non-zero attention scores

In [43]:
all_data[["model_frame_conflict","model_frame_economic","model_frame_human_interest","model_frame_morality","model_frame_attribution"]] = pd.DataFrame(
    all_data["attention_frames"].apply(lambda x: list(x["probabilities"].values())).tolist(),
    index=all_data.index)

all_data["frame_attention_idx"] = all_data["attention_frames"].apply(lambda x: x["explanation"])

all_data[["model_angle_economics","model_angle_geopolitics","model_angle_health","model_angle_jobs","model_angle_politics","model_angle_purchasing_power"]] = pd.DataFrame(
    all_data["attention_angles"].apply(lambda x: list(x["probabilities"].values())).tolist(),
    index=all_data.index)

all_data["angle_attention_idx"] = all_data["attention_angles"].apply(lambda x: x["explanation"])

This is the data needed to populate the database:

In [44]:
df = all_data[["id","source_id","title","body","language","word_count","url","query_source",'angle_health','angle_economics','angle_jobs','angle_politics','angle_geopolitics','angle_purchasing_power',
        'classified_at','ingested_at',
        'sector_agriculture', 'sector_buildings', 'sector_ecosystem', 'sector_energy', 'sector_industry', 'sector_resource','sector_transport', 'sector_water',
        'frame_conflict','frame_economic','frame_human_interest','frame_morality','frame_attribution',
        "model_frame_conflict","model_frame_economic","model_frame_human_interest","model_frame_morality","model_frame_attribution", "frame_attention_idx",
        "model_angle_economics","model_angle_geopolitics","model_angle_health","model_angle_jobs","model_angle_politics","model_angle_purchasing_power", "angle_attention_idx",
        'stage_concept', 'stage_cause', 'stage_consequence', 'stage_solution',
        'domain_climate','domain_biodiversity','domain_ressources',
        'subject','industry','geographic','published_at','outlet']]
        
df.to_csv("datasets/database_input.csv",index=False)

### Data analysis

In [58]:
df = all_data[["id","source_id","title","body","language","word_count","url","query_source",'angle_health','angle_economics','angle_jobs','angle_politics','angle_geopolitics','angle_purchasing_power',
        'classified_at','ingested_at',
        'sector_agriculture', 'sector_buildings', 'sector_ecosystem', 'sector_energy', 'sector_industry', 'sector_resource','sector_transport', 'sector_water',
        'stage_concept', 'stage_cause', 'stage_consequence', 'stage_solution',
        'frame_conflict','frame_economic','frame_human_interest','frame_morality','frame_attribution','subject','industry','geographic','published_at','outlet','frames','angles']]
df = df[df["language"]=="french"]
df["an"] = df["source_id"]
frame_columns = ['frame_conflict','frame_economic','frame_human_interest','frame_morality','frame_attribution']
stage_columns = ['stage_concept', 'stage_cause', 'stage_consequence', 'stage_solution']
sector_columns = ['sector_agriculture', 'sector_buildings', 'sector_ecosystem', 'sector_energy', 'sector_industry', 'sector_resource','sector_transport', 'sector_water']

#### Inter-coder reliability

The annotators worked on separate spreadsheets, their answer format was adjusted before exporting to CSV.
none is used for no frame, else comma-separated list of the frames.

In [59]:
def preprocess_human_labels(label):
    if label == "none":
        return []
    labels = label.split(",")
    return [i.strip() for i in labels]

annotator_1["frames"] = annotator_1["frame_french"].apply(preprocess_human_labels)
annotator_2["frames"] = annotator_2["frame_french"].apply(preprocess_human_labels)

Merging into one dataframe to get Krippendorff alpha:

In [60]:
merged_annotators = annotator_1[['an', 'frames']].merge(annotator_2[['an', 'frames']], on='an', suffixes=('_1', '_2'))

for frame in frame_rename.keys():
    c1 = merged_annotators['frames_1'].apply(lambda x: 1 if frame in x else 0).tolist()
    c2 = merged_annotators['frames_2'].apply(lambda x: 1 if frame in x else 0).tolist()
    
    print(frame,end=": ")
    print(krippendorff.alpha(reliability_data=[c1, c2], level_of_measurement="nominal"))

Conflict: 0.7518656716417911
Economic Consequences: 0.6116983791402396
Human Interest: 0.6670838548185232
Morality: 0.39041666666666675
Responsibility: 0.5142045454545454


Agreement for human interest and conflict is above 0.667 threshold (albeit barely for human interest), agreement is (significantly) below for the three others.

Thus, we decided to resolve all annotator disagreements through discussion by referring to the codebook at all times.
This lead to higher agreement between annotators / more consistency. (As before, the LLM annotations were not seen by the annotators during this process).

The results of the agreed annotation (`annotated`) the LLM annotations (`llm_annotated`) are compared below:

In [61]:
annotated["frames"] = annotated["frame_french"].apply(preprocess_human_labels)

for key,value in frame_rename.items():
    annotated[f"human_{value}"] = annotated["frames"].apply(lambda x: 1 if key in x else 0)

In [62]:
llm_annotated = qc_frame_labels.sample(n=200, random_state=0)[["an","frames"]]
llm_annotated["frames"] = llm_annotated["frames"].apply(lambda x: translate_labels(x,"frames","french"))

In [63]:
all_annotators = annotated[['an', 'frames']].merge(
                    llm_annotated[['an', 'frames']], on='an', suffixes=('_human', '_llm'))

for frame in frame_rename.keys():
    c1 = all_annotators['frames_human'].apply(lambda x: 1 if frame in x else 0).tolist()
    c2 = all_annotators['frames_llm'].apply(lambda x: 1 if frame in x else 0).tolist()
    
    print(frame,end=": \n")

    print(f"\t alpha: {krippendorff.alpha(reliability_data=[c1, c2], level_of_measurement="nominal")}")
    print(f"\t accuracy: {accuracy_score(c1,c2)}")
    print(f"\t precision: {precision_score(c1,c2)}")
    print(f"\t recall: {recall_score(c1,c2)}")
    print(f"\t f1: {f1_score(c1,c2)}")

Conflict: 
	 alpha: 0.6810551558752997
	 accuracy: 0.855
	 precision: 0.7534246575342466
	 recall: 0.8333333333333334
	 f1: 0.7913669064748201
Economic Consequences: 
	 alpha: 0.6866036363636363
	 accuracy: 0.865
	 precision: 0.6533333333333333
	 recall: 0.98
	 f1: 0.784
Human Interest: 
	 alpha: 0.68828125
	 accuracy: 0.9
	 precision: 0.75
	 recall: 0.75
	 f1: 0.75
Morality: 
	 alpha: 0.6163126147390506
	 accuracy: 0.945
	 precision: 0.7692307692307693
	 recall: 0.5555555555555556
	 f1: 0.6451612903225806
Responsibility: 
	 alpha: 0.28255208333333326
	 accuracy: 0.71
	 precision: 0.7098765432098766
	 recall: 0.9126984126984127
	 f1: 0.7986111111111112


#### Distribution of frames

Since the LLM annotations are systematically biased, we deploy PPI (see https://www.science.org/doi/10.1126/science.adi6000)

Assessing distribution of frames:

In [64]:
target_cols = [f"human_{c}" for c in frame_columns]
h_slice = annotated[['an'] + target_cols]
slice_cols = ['an'] + frame_columns + stage_columns
merged_stats = df[slice_cols].merge(h_slice, on='an', how='left')

distribution_results = []
p_frames = pd.Series()

for raw_frame, df_col in frame_rename.items():
    human_col = f"human_{df_col}"
    labeled_mask = merged_stats[human_col].notna().values
    
    Y_labeled = merged_stats.loc[labeled_mask, human_col].to_numpy(dtype=float).flatten()
    Yhat_labeled = merged_stats.loc[labeled_mask, df_col].to_numpy(dtype=float).flatten()
    Yhat_unlabeled = merged_stats.loc[~labeled_mask, df_col].to_numpy(dtype=float).flatten()
    
    corrected_mean = ppi_mean_pointestimate(Y_labeled, Yhat_labeled, Yhat_unlabeled)
    lower_ci, upper_ci = ppi_mean_ci(Y_labeled, Yhat_labeled, Yhat_unlabeled, alpha=0.05)
    
    p_frames[df_col] = corrected_mean

    distribution_results.append({"Frame": raw_frame,
        "LLM %": Yhat_unlabeled.mean() * 100,
        "Corrected %": corrected_mean.astype(float) * 100,
        "Original human %": annotated[human_col].mean(),
        "95% lower": lower_ci * 100,
        "95% upper": upper_ci * 100})
    
pd.DataFrame(distribution_results)

,Frame,LLM %,Corrected %,Original human %,95% lower,95% upper
0,Conflict,36.607143,[33.07058845372376],0.33,[28.28672348673522],[37.8544534207123]
1,Economic Consequences,40.108225,[26.624956377826397],0.25,[22.443769824099675],[30.806142931553122]
2,Human Interest,20.995671,[20.653418920413692],0.20,[16.605623610001867],[24.70121423082552]
3,Morality,4.875541,[7.4753885943351595],0.09,[4.284707617651522],[10.666069571018797]
4,Responsibility,79.740260,[62.50066571735927],0.63,[56.206412028509455],[68.79491940620909]


To analyse significant relationships from the causal chain stages and differences across sectors,
we run a logistic regression for each frame.
The word count is also used as a variable to increase predictive power.

Preparing the columns by merging the annotated dataset with the main (French) dataset

In [65]:
sectors = df[sector_columns].astype(float)
stages = df[stage_columns].astype(float)
df['log_word_count'] = np.log(df['word_count'])

merged = df[['an'] + frame_columns].merge(h_slice, on='an', how='left')

Checking correlations between frames:

In [66]:
annotated[['human_frame_conflict',
       'human_frame_economic', 'human_frame_human_interest',
       'human_frame_morality', 'human_frame_attribution']].corr('pearson')

,human_frame_conflict,human_frame_economic,human_frame_human_interest,human_frame_morality,human_frame_attribution
human_frame_conflict,1.000000,-0.061392,-0.297738,-0.109240,0.427714
human_frame_economic,-0.061392,1.000000,-0.115470,-0.181568,-0.131541
human_frame_human_interest,-0.297738,-0.115470,1.000000,0.017471,-0.160521
human_frame_morality,-0.109240,-0.181568,0.017471,1.000000,0.060071
human_frame_attribution,0.427714,-0.131541,-0.160521,0.060071,1.000000


Given that the frames are not independent variables, and the large amount of statistical tests being run (4 stages, 8 sectors, across 5 frames), we are correcting for multiple comparisons / False Discovery Rate by applying the Benjamini-Hochberg procedure across all tests.

In [67]:
from numpy.linalg import LinAlgError
X_df = pd.concat([df['log_word_count'], stages, sectors], axis=1)
X_df = sm.add_constant(X_df)
X_full = X_df.values
test_vars = list(X_df.columns)

results = []
uncorrected_pvals = []
for raw_frame, df_col in frame_rename.items():
    human_col = f"human_{df_col}"
    labeled_mask = merged[human_col].notna().values
    
    Y_labeled = merged.loc[labeled_mask, human_col].values
    Yhat_labeled = merged.loc[labeled_mask, df_col].astype(float).values
    
    if len(np.unique(Y_labeled)) < 2:
        continue
        
    X_labeled = X_full[labeled_mask]
    X_unlabeled = X_full[~labeled_mask]
    Yhat_unlabeled = merged.loc[~labeled_mask, df_col].astype(float).values
    
    keep_cols = [0]
    for col_idx in range(1, X_labeled.shape[1]):
        col_data = X_labeled[:, col_idx]
        if np.var(col_data) < 1e-9:
            continue
        is_collinear = False
        for kept_idx in keep_cols:
            if np.allclose(col_data, X_labeled[:, kept_idx]) or np.allclose(col_data, 1.0 - X_labeled[:, kept_idx]):
                is_collinear = True
                break
        if not is_collinear:
            keep_cols.append(col_idx)
    
    X_labeled_clean = X_labeled[:, keep_cols]
    X_unlabeled_clean = X_unlabeled[:, keep_cols]
    active_test_vars = [test_vars[idx] for idx in keep_cols]
    try:
        coefs = ppi_logistic_pointestimate(X_labeled_clean, Y_labeled, Yhat_labeled, X_unlabeled_clean, Yhat_unlabeled)
    except LinAlgError: #not enough samples to do PPI (case for one setting)
        continue
    lower_ci, upper_ci = ppi_logistic_ci(X_labeled_clean, Y_labeled, Yhat_labeled, X_unlabeled_clean, Yhat_unlabeled, alpha=0.05)
    
    for i, var in enumerate(active_test_vars):
        coef = coefs[i]
        lower = lower_ci[i]
        upper = upper_ci[i]
        
        se = (upper - lower) / (2 * 1.95996) 
        z_stat = coef / se
        p_val = 2 * (1 - norm.cdf(np.abs(z_stat)))
        
        var_type = "Stage" if var in stage_columns else ("Sector" if var in sector_columns else "Other")
        
        results.append({
            "Type": var_type,
            "Variable": var,
            "Frame": raw_frame,
            "Odds_Ratio": np.exp(coef),
            "PPI_Coef": coef,
            "Raw_P_Value": p_val
        })
        uncorrected_pvals.append(p_val)


adjusting p-values:

In [68]:
analysis_df = pd.DataFrame(results)

reject, pvals_corrected, _, _ = multipletests(uncorrected_pvals, alpha=0.05, method='fdr_bh')
analysis_df['adjusted_p'] = pvals_corrected
analysis_df['Valid'] = reject #null hyp. rejected
analysis_df = analysis_df[analysis_df["Valid"]==True]
analysis_df[analysis_df["Type"]!="Other"]

,Type,Variable,Frame,Odds_Ratio,PPI_Coef,Raw_P_Value,adjusted_p,Valid
3,Stage,stage_cause,Conflict,2.997660,1.097832,0.002368,0.018649,True
4,Stage,stage_consequence,Conflict,0.503126,-0.686914,0.008889,0.041480,True
38,Sector,sector_industry,Human Interest,0.336639,-1.088744,0.010554,0.045462,True
45,Stage,stage_cause,Responsibility,4.139218,1.420507,0.000562,0.009738,True
53,Sector,sector_resource,Responsibility,0.342944,-1.070188,0.005899,0.033037,True


### Scaling logits for model deployment

Since the LLM labels are structurally biased, this also affects the RoBERTa model.
To account for the over-/underpredicting of the LLM, we apply Platt scaling to correct the logit outputs of the RoBERTa model.

Thus, we fit a logistic regression using the raw logits to the annotated dataset, and use the coefficient+intercept as weight+bias for the logit that is passed into the sigmoid function in the model.

To get the raw logits, run `deployment.py` with `return_logits=True` for the human annotated dataset.

This correction is only for the frame model, since we don't have human annotation for the angles, due to the project's focus.

(Note that this has to be run to create the necessary attention_frame.pkl that is loaded in the beginning of this notebook to be stored in the database. So for the complete procedure, this code block has to be run after storing the data for the model, but it's at the end of the notebook for better overview)

In [69]:
sampled_with_model_logits = pd.read_csv("datasets/attention/sampled_with_logits.csv")
sampled_with_model_logits = sampled_with_model_logits.merge(annotated,on="an")

print("deployment_calibrations = {")

for raw_frame, df_col in frame_rename.items():
    human_col = f"human_{df_col}" 
    logit_col = f"roberta_logit_{raw_frame}" 

    valid_data = sampled_with_model_logits[[human_col, logit_col]].dropna()
    y = valid_data[human_col].values
    X = valid_data[logit_col].values.reshape(-1, 1)

    lr = LogisticRegression(solver='lbfgs')
    lr.fit(X, y)
    
    W = lr.coef_[0][0]
    b = lr.intercept_[0]
    print(f"    '{raw_frame}': {{'W': {W:.4f}, 'b': {b:.4f}}},")
print("}")

deployment_calibrations = {
    'Conflict': {'W': 1.1204, 'b': -0.2250},
    'Economic Consequences': {'W': 0.7453, 'b': -1.7384},
    'Human Interest': {'W': 0.6387, 'b': -0.5708},
    'Morality': {'W': 1.4947, 'b': 2.1579},
    'Responsibility': {'W': 0.4394, 'b': -0.2494},
}
